In [212]:
import numpy as np
import pandas as pd

In [213]:
match_data = pd.read_csv('../data/ProcessedDataForTestModel/historical_match_data_2022_present.csv')
rank_data = pd.read_csv('../data/ProcessedDataForTestModel/FIFA_rankings_2021_2026.csv')


In [214]:
# =========================
# 1. Clean unnecessary columns
# =========================

match_data = match_data.drop(columns=["Unnamed: 0"], errors="ignore")
rank_data = rank_data.drop(columns=["Unnamed: 0"], errors="ignore")

# =========================
# 2. Convert dates
# =========================

match_data["date"] = pd.to_datetime(match_data["date"])
rank_data["rank_date"] = pd.to_datetime(rank_data["rank_date"])

# =========================
# 3. Fix team-name differences
# =========================

TEAM_NAME_FIXES = {
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "Kyrgyzstan": "Kyrgyz Republic",
    "North Korea": "Korea DPR",
    "South Korea": "Korea Republic",
    "Taiwan": "Chinese Taipei",
    "United States": "USA",
    "United States Virgin Islands": "US Virgin Islands",
    "Saint Kitts and Nevis": "St Kitts and Nevis",
    "Saint Lucia": "St Lucia",
    "Saint Vincent and the Grenadines": "St Vincent and the Grenadines",
}

match_data["home_team_rank_name"] = match_data["home_team"].replace(TEAM_NAME_FIXES)
match_data["away_team_rank_name"] = match_data["away_team"].replace(TEAM_NAME_FIXES)

# =========================
# 4. Keep only useful ranking columns
# =========================

ranking_data = rank_data[
    [
        "rank_date",
        "country",
        "country_code",
        "rank",
        "previous_rank",
        "rank_change",
        "total_points",
        "previous_points"
    ]
].copy()

ranking_data = ranking_data.rename(columns={
    "country": "team_rank_name"
})

# =========================
# 5. Create match_id
# =========================

match_data = match_data.reset_index(drop=True)
match_data["match_id"] = match_data.index

# =========================
# 6. Function for time-aware merge
# =========================

def add_ranking_to_matches(matches_df, team_col, prefix):
    """
    Adds the latest FIFA ranking available before or on the match date.
    
    For example, prefix='home' creates:
        home_rank
        home_total_points
        home_rank_date_used
    """

    left = matches_df[
        [
            "match_id",
            "date",
            team_col
        ]
    ].copy()

    left = left.rename(columns={
        team_col: "team_rank_name"
    })

    right = ranking_data.copy()

    left = left.sort_values(["date", "team_rank_name"])
    right = right.sort_values(["rank_date", "team_rank_name"])

    merged = pd.merge_asof(
        left,
        right,
        left_on="date",
        right_on="rank_date",
        by="team_rank_name",
        direction="backward"
    )

    merged = merged.rename(columns={
        "rank_date": f"{prefix}_rank_date_used",
        "country_code": f"{prefix}_country_code",
        "rank": f"{prefix}_rank",
        "previous_rank": f"{prefix}_previous_rank",
        "rank_change": f"{prefix}_rank_change",
        "total_points": f"{prefix}_total_points",
        "previous_points": f"{prefix}_previous_points"
    })

    return merged[
        [
            "match_id",
            f"{prefix}_rank_date_used",
            f"{prefix}_country_code",
            f"{prefix}_rank",
            f"{prefix}_previous_rank",
            f"{prefix}_rank_change",
            f"{prefix}_total_points",
            f"{prefix}_previous_points"
        ]
    ]

# =========================
# 7. Add home and away rankings
# =========================

home_ranking = add_ranking_to_matches(
    match_data,
    team_col="home_team_rank_name",
    prefix="home"
)

away_ranking = add_ranking_to_matches(
    match_data,
    team_col="away_team_rank_name",
    prefix="away"
)

final_data = match_data.merge(home_ranking, on="match_id", how="left")
final_data = final_data.merge(away_ranking, on="match_id", how="left")

# =========================
# 8. Add useful ranking features
# =========================

final_data["rank_advantage"] = final_data["away_rank"] - final_data["home_rank"]
final_data["points_diff"] = final_data["home_total_points"] - final_data["away_total_points"]

# Lower FIFA rank is better.
# So if advantage < 0, home team is better ranked.

# =========================
# 9. Add target variable
# =========================

def get_match_result(row):
    if row["home_score"] > row["away_score"]:
        return 1      # home win
    elif row["home_score"] < row["away_score"]:
        return -1     # away win
    else:
        return 0      # draw

final_data["result"] = final_data.apply(get_match_result, axis=1)

# =========================
# 10. Check missing rankings
# =========================

missing_rankings = final_data[
    final_data["home_rank"].isna() | final_data["away_rank"].isna()
][
    [
        "date",
        "home_team",
        "away_team",
        "home_team_rank_name",
        "away_team_rank_name",
        "home_rank",
        "away_rank"
    ]
]

print("Number of matches with missing rankings:", len(missing_rankings))
display(missing_rankings.head(30))

## Drop the helper columns:

final_data = final_data.drop(
    columns=[
        "home_team_rank_name",
        "away_team_rank_name",
        "home_country_code",
        "away_country_code"
    ],
    errors="ignore"
)

# =========================
# 12. Save final CSV
# =========================

final_data.to_csv("../data/ProcessedDataForTestModel/matches_with_fifa_rankings.csv", index=False)

print("Saved: matches_with_fifa_rankings.csv")
display(final_data.head())

Number of matches with missing rankings: 1


,date,home_team,away_team,home_team_rank_name,away_team_rank_name,home_rank,away_rank
34,2022-03-17,Cook Islands,Solomon Islands,Cook Islands,Solomon Islands,NaN,142.0


Saved: matches_with_fifa_rankings.csv


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,home_previous_points,away_rank_date_used,away_rank,away_previous_rank,away_rank_change,away_total_points,away_previous_points,rank_advantage,points_diff,result
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,1412.13,2021-12-23,14.0,14.0,0.0,1638.30,1638.76,-43.0,-226.17,-1
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,1541.33,2021-12-23,5.0,5.0,0.0,1750.51,1750.51,-19.0,-207.09,-1
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,1454.52,2021-12-23,17.0,17.0,0.0,1596.66,1596.66,-26.0,-142.14,-1
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,1448.74,2021-12-23,2.0,2.0,0.0,1826.35,1826.35,-44.0,-378.08,0
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,1441.17,2021-12-23,79.0,78.0,-1.0,1306.01,1307.09,28.0,128.70,1


In [215]:
final_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1065 entries, 0 to 1064
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  1065 non-null   datetime64[us]
 1   home_team             1065 non-null   str           
 2   away_team             1065 non-null   str           
 3   home_score            1065 non-null   float64       
 4   away_score            1065 non-null   float64       
 5   tournament            1065 non-null   str           
 6   city                  1065 non-null   str           
 7   country               1065 non-null   str           
 8   neutral               1065 non-null   int64         
 9   year                  1065 non-null   int64         
 10  is_world_cup          1065 non-null   int64         
 11  is_qualifier          1065 non-null   int64         
 12  match_id              1065 non-null   int64         
 13  home_rank_date_used   1064 no

In [216]:
final_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,home_previous_points,away_rank_date_used,away_rank,away_previous_rank,away_rank_change,away_total_points,away_previous_points,rank_advantage,points_diff,result
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,1412.13,2021-12-23,14.0,14.0,0.0,1638.30,1638.76,-43.0,-226.17,-1
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,1541.33,2021-12-23,5.0,5.0,0.0,1750.51,1750.51,-19.0,-207.09,-1
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,1454.52,2021-12-23,17.0,17.0,0.0,1596.66,1596.66,-26.0,-142.14,-1
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,1448.74,2021-12-23,2.0,2.0,0.0,1826.35,1826.35,-44.0,-378.08,0
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,1441.17,2021-12-23,79.0,78.0,-1.0,1306.01,1307.09,28.0,128.70,1


In [217]:
##Adding the ELO ratings
elo_data = pd.read_csv('../data/raw_data/elo_ratings_wc2026.csv')
elo_data.tail(40)

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
4643,2026,2026-05-27,Ecuador,9,EC,1933,9,1934,64,1525,...,183,211,204,183,254,161,719,903,CONMEBOL,0
4644,2026,2026-05-27,Croatia,10,HR,1930,4,2015,12,1881,...,156,167,75,209,83,106,694,401,UEFA,0
4645,2026,2026-05-27,Germany,11,DE,1923,1,2223,8,1911,...,469,415,161,604,224,217,2352,1232,UEFA,0
4646,2026,2026-05-27,Norway,12,NO,1912,6,1944,38,1623,...,425,409,52,334,355,197,1373,1443,UEFA,0
4647,2026,2026-05-27,Japan,13,JP,1904,9,1913,62,1467,...,331,207,297,408,257,170,1517,985,AFC,0
4648,2026,2026-05-27,Turkey,14,TR,1902,9,1902,42,1611,...,290,300,79,263,250,156,947,969,UEFA,0
4649,2026,2026-05-27,Uruguay,15,UY,1892,1,2104,12,1876,...,336,432,251,446,321,252,1610,1254,CONMEBOL,0
4650,2026,2026-05-27,Switzerland,16,CH,1889,9,1933,27,1690,...,425,392,78,319,373,203,1335,1482,UEFA,0
4651,2026,2026-05-27,Senegal,17,SN,1878,17,1878,56,1590,...,221,259,224,319,192,193,967,669,CAF,0
4652,2026,2026-05-27,Belgium,19,BE,1867,1,2157,24,1755,...,415,375,86,390,302,184,1599,1359,UEFA,0


In [218]:
elo_current_2026_for_test_dataset = pd.read_csv('../data/elo_2026_current.csv')
elo_current_2026_for_test_dataset

,elo_date,elo_rank,team,elo_rating,average_rank,average_rating
0,2026-06-11,1,Spain,2157,7,1946
1,2026-06-11,2,Argentina,2115,5,1987
2,2026-06-11,3,France,2063,16,1795
3,2026-06-11,4,England,2024,4,1983
4,2026-06-11,5,Brazil,1991,4,1998
...,...,...,...,...,...,...
238,2026-06-11,240,Niue,496,225,496
239,2026-06-11,241,Northern Mariana Islands,432,233,512
240,2026-06-11,242,Cocos Islands,422,235,423
241,2026-06-11,243,Palau,402,237,402


In [219]:
# =========================
# 1. Load data
# =========================

final_data["date"] = pd.to_datetime(final_data["date"])
elo_data["snapshot_date"] = pd.to_datetime(elo_data["snapshot_date"])

# =========================
# 2. Keep useful ELO columns
# =========================

elo_data = elo_data[
    [
        "snapshot_date",
        "country",
        "rank",
        "rating",
        "rank_avg",
        "rating_avg",
        "matches_total",
        "wins",
        "losses",
        "draws",
        "goals_for",
        "goals_against",
        "confederation",
        "is_host"
    ]
].copy()

elo_data = elo_data.rename(columns={
    "snapshot_date": "elo_date",
    "country": "team_elo_name",
    "rank": "elo_rank",
    "rating": "elo_rating",
    "rank_avg": "elo_avg_rank",
    "rating_avg": "elo_avg_rating"
})

# Keep from 2021 because early 2022 matches need previous ELO snapshot
elo_data = elo_data[elo_data["elo_date"] >= "2021-01-01"].copy()

# =========================
# 3. Fix team-name differences
# =========================

TEAM_NAME_FIXES_ELO = {
    "USA": "United States",
    "United States": "United States",

    "IR Iran": "Iran",
    "Iran": "Iran",

    "Korea Republic": "South Korea",
    "South Korea": "South Korea",

    "Korea DPR": "North Korea",
    "North Korea": "North Korea",

    "Congo DR": "DR Congo",
    "DR Congo": "DR Congo",

    "Côte d'Ivoire": "Ivory Coast",
    "Ivory Coast": "Ivory Coast",

    "Cabo Verde": "Cape Verde",
    "Cape Verde": "Cape Verde",

    "Kyrgyz Republic": "Kyrgyzstan",
    "Kyrgyzstan": "Kyrgyzstan",
}

final_data["home_team_elo_name"] = final_data["home_team"].replace(TEAM_NAME_FIXES_ELO)
final_data["away_team_elo_name"] = final_data["away_team"].replace(TEAM_NAME_FIXES_ELO)

# =========================
# 4. Make sure match_id exists
# =========================

if "match_id" not in final_data.columns:
    final_data = final_data.reset_index(drop=True)
    final_data["match_id"] = final_data.index

# =========================
# 5. Function for time-aware ELO merge
# =========================

def add_elo_to_matches(matches_df, team_col, prefix):
    left = matches_df[
        [
            "match_id",
            "date",
            team_col
        ]
    ].copy()

    left = left.rename(columns={
        team_col: "team_elo_name"
    })

    right = elo_data.copy()

    left = left.sort_values(["date", "team_elo_name"])
    right = right.sort_values(["elo_date", "team_elo_name"])

    merged = pd.merge_asof(
        left,
        right,
        left_on="date",
        right_on="elo_date",
        by="team_elo_name",
        direction="backward"
    )

    merged = merged.rename(columns={
        "elo_date": f"{prefix}_elo_date_used",
        "elo_rank": f"{prefix}_elo_rank",
        "elo_rating": f"{prefix}_elo",
        "elo_avg_rank": f"{prefix}_elo_avg_rank",
        "elo_avg_rating": f"{prefix}_elo_avg_rating",
        "matches_total": f"{prefix}_elo_matches_total",
        "wins": f"{prefix}_elo_wins",
        "losses": f"{prefix}_elo_losses",
        "draws": f"{prefix}_elo_draws",
        "goals_for": f"{prefix}_elo_goals_for",
        "goals_against": f"{prefix}_elo_goals_against",
        "confederation": f"{prefix}_confederation",
        "is_host": f"{prefix}_is_host"
    })

    return merged[
        [
            "match_id",
            f"{prefix}_elo_date_used",
            f"{prefix}_elo_rank",
            f"{prefix}_elo",
            f"{prefix}_elo_avg_rank",
            f"{prefix}_elo_avg_rating",
            f"{prefix}_elo_matches_total",
            f"{prefix}_elo_wins",
            f"{prefix}_elo_losses",
            f"{prefix}_elo_draws",
            f"{prefix}_elo_goals_for",
            f"{prefix}_elo_goals_against",
            f"{prefix}_confederation",
            f"{prefix}_is_host"
        ]
    ]

# =========================
# 6. Add home and away ELO
# =========================

home_elo = add_elo_to_matches(
    final_data,
    team_col="home_team_elo_name",
    prefix="home"
)

away_elo = add_elo_to_matches(
    final_data,
    team_col="away_team_elo_name",
    prefix="away"
)

final_data = final_data.merge(home_elo, on="match_id", how="left")
final_data = final_data.merge(away_elo, on="match_id", how="left")

# =========================
# 7. Add useful ELO features
# =========================

final_data["elo_advantage"] = final_data["home_elo"] - final_data["away_elo"]

# Lower rank is better, so away - home means positive is good for home
final_data["elo_rank_advantage"] = final_data["away_elo_rank"] - final_data["home_elo_rank"]

final_data["elo_avg_rating_advantage"] = (
    final_data["home_elo_avg_rating"] - final_data["away_elo_avg_rating"]
)

final_data["elo_win_rate_home"] = (
    final_data["home_elo_wins"] / final_data["home_elo_matches_total"]
)

final_data["elo_win_rate_away"] = (
    final_data["away_elo_wins"] / final_data["away_elo_matches_total"]
)

final_data["elo_win_rate_advantage"] = (
    final_data["elo_win_rate_home"] - final_data["elo_win_rate_away"]
)

# =========================
# 8. Check missing ELO
# =========================

missing_elo = final_data[
    final_data["home_elo"].isna() | final_data["away_elo"].isna()
][
    [
        "date",
        "home_team",
        "away_team",
        "home_team_elo_name",
        "away_team_elo_name",
        "home_elo",
        "away_elo"
    ]
]

print("Number of matches with missing ELO:", len(missing_elo))
display(missing_elo.head(50))

# =========================
# 9. Drop helper columns
# =========================

final_data = final_data.drop(
    columns=[
        "home_team_elo_name",
        "away_team_elo_name",
        "home_confederation",
        "away_confederation"
    ],
    errors="ignore"
)

# =========================
# 10. Save
# =========================

final_data.to_csv("../data/matches_with_fifa_rankings_and_elo.csv", index=False)

print("Saved: ../data/matches_with_fifa_rankings_and_elo.csv")
display(final_data.head())

Number of matches with missing ELO: 936


,date,home_team,away_team,home_team_elo_name,away_team_elo_name,home_elo,away_elo
0,2022-01-27,Jamaica,Mexico,Jamaica,Mexico,NaN,1839.0
1,2022-01-27,Chile,Argentina,Chile,Argentina,NaN,2101.0
4,2022-01-27,Saudi Arabia,Oman,Saudi Arabia,Oman,1629.0,NaN
5,2022-01-27,Japan,China PR,Japan,China PR,1760.0,NaN
6,2022-01-27,Australia,Vietnam,Australia,Vietnam,1735.0,NaN
8,2022-01-27,Lebanon,South Korea,Lebanon,South Korea,NaN,1788.0
9,2022-01-27,Costa Rica,Panama,Costa Rica,Panama,NaN,1627.0
10,2022-01-27,Honduras,Canada,Honduras,Canada,NaN,1784.0
11,2022-01-27,United States,El Salvador,United States,El Salvador,1858.0,NaN
12,2022-01-27,United Arab Emirates,Syria,United Arab Emirates,Syria,NaN,NaN


Saved: ../data/matches_with_fifa_rankings_and_elo.csv


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_elo_draws,away_elo_goals_for,away_elo_goals_against,away_is_host,elo_advantage,elo_rank_advantage,elo_avg_rating_advantage,elo_win_rate_home,elo_win_rate_away,elo_win_rate_advantage
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,216.0,1719.0,988.0,1.0,NaN,NaN,NaN,NaN,0.520462,NaN
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,265.0,1991.0,1112.0,0.0,NaN,NaN,NaN,NaN,0.537879,NaN
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,235.0,1537.0,1221.0,0.0,-118.0,-20.0,-119.0,0.351678,0.435501,-0.083823
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,211.0,2201.0,909.0,0.0,-300.0,-15.0,-489.0,0.299270,0.634218,-0.334948
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.482940,NaN,NaN


In [220]:
final_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_elo_draws,away_elo_goals_for,away_elo_goals_against,away_is_host,elo_advantage,elo_rank_advantage,elo_avg_rating_advantage,elo_win_rate_home,elo_win_rate_away,elo_win_rate_advantage
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,216.0,1719.0,988.0,1.0,NaN,NaN,NaN,NaN,0.520462,NaN
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,265.0,1991.0,1112.0,0.0,NaN,NaN,NaN,NaN,0.537879,NaN
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,235.0,1537.0,1221.0,0.0,-118.0,-20.0,-119.0,0.351678,0.435501,-0.083823
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,211.0,2201.0,909.0,0.0,-300.0,-15.0,-489.0,0.299270,0.634218,-0.334948
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.482940,NaN,NaN


In [221]:
nan_counts = final_data.isna().sum()

nan_counts = nan_counts[nan_counts > 0]

print(nan_counts)

home_rank_date_used           1
home_rank                     1
home_previous_rank            1
home_rank_change              1
home_total_points             1
home_previous_points          1
rank_advantage                1
points_diff                   1
home_elo_date_used          694
home_elo_rank               694
home_elo                    694
home_elo_avg_rank           694
home_elo_avg_rating         694
home_elo_matches_total      694
home_elo_wins               694
home_elo_losses             694
home_elo_draws              694
home_elo_goals_for          694
home_elo_goals_against      694
home_is_host                694
away_elo_date_used          720
away_elo_rank               720
away_elo                    720
away_elo_avg_rank           720
away_elo_avg_rating         720
away_elo_matches_total      720
away_elo_wins               720
away_elo_losses             720
away_elo_draws              720
away_elo_goals_for          720
away_elo_goals_against      720
away_is_

In [222]:
missing_home = final_data[final_data["home_elo"].isna()]["home_team"].value_counts()
missing_away = final_data[final_data["away_elo"].isna()]["away_team"].value_counts()

print("Missing home ELO teams:")
display(missing_home.head(50))

print("Missing away ELO teams:")
display(missing_away.head(50))

Missing home ELO teams:


home_team
United Arab Emirates    13
Bolivia                 12
Chile                   11
Venezuela               11
Peru                    11
Costa Rica              10
Oman                    10
Jamaica                  9
China PR                 9
Wales                    9
Cameroon                 9
Indonesia                9
Bahrain                  9
Honduras                 8
Poland                   8
Nigeria                  8
Kuwait                   8
Palestine                8
Kyrgyzstan               8
El Salvador              7
North Korea              7
Italy                    6
Mali                     6
Czech Republic           6
Lebanon                  5
Syria                    5
Vietnam                  5
Papua New Guinea         5
Denmark                  5
Serbia                   5
Taiwan                   5
Ethiopia                 5
Rwanda                   5
Equatorial Guinea        5
Botswana                 5
Burundi                  5
Gabon             

Missing away ELO teams:


away_team
Oman                     12
Peru                     12
Bolivia                  12
United Arab Emirates     12
China PR                 11
Chile                    11
Venezuela                11
Indonesia                11
Costa Rica               10
El Salvador               9
Jamaica                   9
Honduras                  8
Palestine                 8
North Korea               8
Kyrgyzstan                8
Kuwait                    8
North Macedonia           7
Cameroon                  7
Poland                    7
Bahrain                   7
New Caledonia             6
Nigeria                   6
Mali                      6
Serbia                    6
Denmark                   6
Zimbabwe                  6
Lesotho                   6
Somalia                   6
São Tomé and Príncipe     6
Gabon                     6
Suriname                  6
Vietnam                   5
Syria                     5
Tahiti                    5
Czech Republic            5
Lebanon   

In [223]:
missing_elo_teams = pd.concat([
    final_data[final_data["home_elo"].isna()]["home_team"],
    final_data[final_data["away_elo"].isna()]["away_team"]
]).value_counts()

display(missing_elo_teams.head(80))

United Arab Emirates    25
Bolivia                 24
Peru                    23
Chile                   22
Venezuela               22
                        ..
Yemen                    8
Singapore                8
Nepal                    8
Myanmar                  8
Hong Kong                8
Name: count, Length: 80, dtype: int64

In [224]:
elo_teams = set(elo_data["team_elo_name"].unique())

missing_team_names = set(missing_elo_teams.index)

teams_not_in_elo = sorted([team for team in missing_team_names if team not in elo_teams])
teams_in_elo = sorted([team for team in missing_team_names if team in elo_teams])

print("Teams NOT found in ELO dataset:")
print(teams_not_in_elo)

print("\nTeams found in ELO dataset but still missing:")
print(teams_in_elo)

Teams NOT found in ELO dataset:
['Afghanistan', 'Albania', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Armenia', 'Aruba', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Botswana', 'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China PR', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Cuba', 'Cyprus', 'Czech Republic', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'El Salvador', 'Equatorial Guinea', 'Estonia', 'Eswatini', 'Ethiopia', 'Faroe Islands', 'Fiji', 'Finland', 'Gabon', 'Gambia', 'Georgia', 'Gibraltar', 'Greece', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Honduras', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Israel', 'Italy', 'Jamaica', 'Kazakhstan', 'Kenya', 'Kosovo', 'Kuwait', 'Kyrgyzstan', 'Laos', '

In [225]:
import pandas as pd

data = pd.read_csv("../data/matches_with_fifa_rankings_and_elo.csv")

data["date"] = pd.to_datetime(data["date"])
data["home_elo_date_used"] = pd.to_datetime(data["home_elo_date_used"])
data["away_elo_date_used"] = pd.to_datetime(data["away_elo_date_used"])

bad_home = data[
    data["home_elo_date_used"] > data["date"]
][
    ["date", "home_team", "away_team", "home_elo_date_used", "home_elo"]
]

bad_away = data[
    data["away_elo_date_used"] > data["date"]
][
    ["date", "home_team", "away_team", "away_elo_date_used", "away_elo"]
]

print("Bad home ELO rows:", len(bad_home))
print("Bad away ELO rows:", len(bad_away))

display(bad_home.head())
display(bad_away.head())

Bad home ELO rows: 0
Bad away ELO rows: 0


,date,home_team,away_team,home_elo_date_used,home_elo


,date,home_team,away_team,away_elo_date_used,away_elo


In [226]:
import pandas as pd
import re

# =========================
# 1. Load data
# =========================

data = pd.read_csv("../data/matches_with_fifa_rankings_and_elo.csv")
elo_hist = pd.read_csv("../data/raw_data/eloratings.csv")

data["date"] = pd.to_datetime(data["date"])
elo_hist["date"] = pd.to_datetime(elo_hist["date"], format="mixed", errors="coerce")

# =========================
# 2. Clean names
# =========================

def clean_team_name(name):
    if pd.isna(name):
        return name

    name = str(name)
    name = name.replace("\xa0", " ")
    name = re.sub(r"\s+", " ", name)
    name = name.strip()

    return name

data["home_team_elo_name_fill"] = data["home_team"].apply(clean_team_name)
data["away_team_elo_name_fill"] = data["away_team"].apply(clean_team_name)
elo_hist["team_elo_name"] = elo_hist["team"].apply(clean_team_name)

# =========================
# 3. Fix name mismatches
# =========================

TEAM_NAME_FIXES_ELO = {
    "American Samoa": "Eastern Samoa",
    "China PR": "China",
    "Czech Republic": "Czechia",
    "DR Congo": "Democratic Republic of Congo",
    "Macau": "Macao",
    "Republic of Ireland": "Ireland",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Timor-Leste": "East Timor",
    "United States Virgin Islands": "US Virgin Islands",

    "USA": "United States",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "IR Iran": "Iran",
    "Congo DR": "Democratic Republic of Congo",
    "Côte d'Ivoire": "Ivory Coast",
    "Cabo Verde": "Cape Verde",
    "Kyrgyz Republic": "Kyrgyzstan",
    "UAE": "United Arab Emirates",
}

data["home_team_elo_name_fill"] = data["home_team_elo_name_fill"].replace(TEAM_NAME_FIXES_ELO)
data["away_team_elo_name_fill"] = data["away_team_elo_name_fill"].replace(TEAM_NAME_FIXES_ELO)

# =========================
# 4. Prepare historical ELO
# =========================

elo_hist = elo_hist[
    [
        "date",
        "team_elo_name",
        "rating",
        "change"
    ]
].copy()

elo_hist = elo_hist.rename(columns={
    "date": "elo_date",
    "rating": "elo_rating",
    "change": "elo_change"
})

elo_hist = elo_hist.dropna(subset=["elo_date", "elo_rating"])

elo_hist = elo_hist.sort_values(
    ["elo_date", "team_elo_name"]
).drop_duplicates(
    ["team_elo_name", "elo_date"],
    keep="last"
)

# =========================
# 5. Make sure match_id exists
# =========================

if "match_id" not in data.columns:
    data = data.reset_index(drop=True)
    data["match_id"] = data.index

# =========================
# 6. Function to get historical ELO
# =========================

def get_historical_elo(matches_df, team_col, prefix):
    left = matches_df[
        [
            "match_id",
            "date",
            team_col
        ]
    ].copy()

    left = left.rename(columns={
        team_col: "team_elo_name"
    })

    left = left.sort_values(["date", "team_elo_name"]).reset_index(drop=True)
    right = elo_hist.sort_values(["elo_date", "team_elo_name"]).reset_index(drop=True)

    # First try: latest available ELO before match date
    previous_elo = pd.merge_asof(
        left,
        right,
        left_on="date",
        right_on="elo_date",
        by="team_elo_name",
        direction="backward",
        allow_exact_matches=False
    )

    previous_elo["elo_source_fill"] = "historical_previous"

    # Fallback: if no previous ELO exists, take earliest available future ELO
    future_elo = pd.merge_asof(
        left,
        right,
        left_on="date",
        right_on="elo_date",
        by="team_elo_name",
        direction="forward",
        allow_exact_matches=True
    )

    missing_mask = previous_elo["elo_rating"].isna()

    previous_elo.loc[missing_mask, "elo_date"] = future_elo.loc[missing_mask, "elo_date"]
    previous_elo.loc[missing_mask, "elo_rating"] = future_elo.loc[missing_mask, "elo_rating"]
    previous_elo.loc[missing_mask, "elo_change"] = future_elo.loc[missing_mask, "elo_change"]

    previous_elo.loc[
        missing_mask & future_elo["elo_rating"].notna(),
        "elo_source_fill"
    ] = "historical_future_fallback"

    previous_elo = previous_elo.rename(columns={
        "elo_date": f"{prefix}_elo_date_fill",
        "elo_rating": f"{prefix}_elo_fill",
        "elo_change": f"{prefix}_elo_change_fill",
        "elo_source_fill": f"{prefix}_elo_source_fill"
    })

    return previous_elo[
        [
            "match_id",
            f"{prefix}_elo_date_fill",
            f"{prefix}_elo_fill",
            f"{prefix}_elo_change_fill",
            f"{prefix}_elo_source_fill"
        ]
    ]

# =========================
# 7. Get fill values for home and away
# =========================

home_fill = get_historical_elo(
    data,
    team_col="home_team_elo_name_fill",
    prefix="home"
)

away_fill = get_historical_elo(
    data,
    team_col="away_team_elo_name_fill",
    prefix="away"
)

data = data.merge(home_fill, on="match_id", how="left")
data = data.merge(away_fill, on="match_id", how="left")

# =========================
# 8. Fill only missing ELO values
# =========================

home_missing_before = data["home_elo"].isna().sum()
away_missing_before = data["away_elo"].isna().sum()

data["home_elo_was_filled"] = data["home_elo"].isna().astype(int)
data["away_elo_was_filled"] = data["away_elo"].isna().astype(int)

data["home_elo"] = data["home_elo"].fillna(data["home_elo_fill"])
data["away_elo"] = data["away_elo"].fillna(data["away_elo_fill"])

data["home_elo_date_used"] = data["home_elo_date_used"].fillna(data["home_elo_date_fill"])
data["away_elo_date_used"] = data["away_elo_date_used"].fillna(data["away_elo_date_fill"])

# Recompute advantage after filling
data["elo_advantage"] = data["home_elo"] - data["away_elo"]

home_missing_after = data["home_elo"].isna().sum()
away_missing_after = data["away_elo"].isna().sum()

print("Home ELO missing before:", home_missing_before)
print("Home ELO missing after:", home_missing_after)

print("Away ELO missing before:", away_missing_before)
print("Away ELO missing after:", away_missing_after)

print("ELO advantage missing:", data["elo_advantage"].isna().sum())

# =========================
# 9. Check no future leakage after filling
# =========================

data["home_elo_date_used"] = pd.to_datetime(data["home_elo_date_used"])
data["away_elo_date_used"] = pd.to_datetime(data["away_elo_date_used"])

bad_home = data[
    data["home_elo_date_used"] > data["date"]
]

bad_away = data[
    data["away_elo_date_used"] > data["date"]
]

print("Bad home ELO date rows:", len(bad_home))
print("Bad away ELO date rows:", len(bad_away))

# =========================
# 10. Drop temporary fill columns
# =========================

temp_cols = [
    "home_team_elo_name_fill",
    "away_team_elo_name_fill",

    "home_elo_date_fill",
    "home_elo_fill",
    "home_elo_change_fill",
    "home_elo_source_fill",

    "away_elo_date_fill",
    "away_elo_fill",
    "away_elo_change_fill",
    "away_elo_source_fill",
]

data = data.drop(columns=temp_cols, errors="ignore")

# =========================
# 11. Save
# =========================

data.to_csv("../data/matches_with_fifa_rankings_and_elo_filled.csv", index=False)

print("Saved: ../data/matches_with_fifa_rankings_and_elo_filled.csv")
display(data.head())

Home ELO missing before: 694
Home ELO missing after: 0
Away ELO missing before: 720
Away ELO missing after: 0
ELO advantage missing: 0
Bad home ELO date rows: 7
Bad away ELO date rows: 4
Saved: ../data/matches_with_fifa_rankings_and_elo_filled.csv


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_elo_goals_against,away_is_host,elo_advantage,elo_rank_advantage,elo_avg_rating_advantage,elo_win_rate_home,elo_win_rate_away,elo_win_rate_advantage,home_elo_was_filled,away_elo_was_filled
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,988.0,1.0,-261.0,NaN,NaN,NaN,0.520462,NaN,1,0
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,1112.0,0.0,-288.0,NaN,NaN,NaN,0.537879,NaN,1,0
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,1221.0,0.0,-118.0,-20.0,-119.0,0.351678,0.435501,-0.083823,0,0
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,909.0,0.0,-300.0,-15.0,-489.0,0.299270,0.634218,-0.334948,0,0
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,NaN,NaN,133.0,NaN,NaN,0.482940,NaN,NaN,0,1


In [227]:
data["date"] = pd.to_datetime(data["date"])
data["home_elo_date_used"] = pd.to_datetime(data["home_elo_date_used"])
data["away_elo_date_used"] = pd.to_datetime(data["away_elo_date_used"])

bad_rows = data[
    (data["home_elo_date_used"] > data["date"]) |
    (data["away_elo_date_used"] > data["date"])
][
    [
        "date",
        "home_team",
        "away_team",
        "home_elo",
        "away_elo",
        "home_elo_date_used",
        "away_elo_date_used",
        "home_elo_was_filled",
        "away_elo_was_filled",
        "elo_advantage"
    ]
]

print("Number of bad rows:", len(bad_rows))
display(bad_rows)

Number of bad rows: 11


,date,home_team,away_team,home_elo,away_elo,home_elo_date_used,away_elo_date_used,home_elo_was_filled,away_elo_was_filled,elo_advantage
34,2022-03-17,Cook Islands,Solomon Islands,622.0,1196.0,2024-09-09,2017-12-15,1,1,-574.0
469,2024-09-06,Cook Islands,Tonga,622.0,546.0,2024-09-09,2017-12-15,1,1,76.0
470,2024-09-06,Samoa,American Samoa,677.0,389.0,2023-11-17,2024-09-09,1,1,288.0
609,2025-03-22,Moldova,Norway,0.0,1806.0,2025-12-13,2024-12-31,1,0,-1806.0
636,2025-03-25,Moldova,Estonia,0.0,1357.0,2025-12-13,2024-03-21,1,1,-1357.0
710,2025-06-09,Italy,Moldova,1943.0,0.0,2024-09-09,2025-12-13,1,1,1943.0
781,2025-09-05,Moldova,Israel,0.0,1629.0,2025-12-13,2025-06-10,1,1,-1629.0
847,2025-09-09,Norway,Moldova,1806.0,0.0,2024-12-31,2025-12-13,0,1,1806.0
966,2025-10-14,Estonia,Moldova,1356.0,0.0,2025-09-09,2025-12-13,1,1,1356.0
992,2025-11-13,Moldova,Italy,0.0,1943.0,2025-12-13,2024-09-09,1,1,-1943.0


In [228]:
data_clean = data[
    ~(
        (data["home_elo_date_used"] > data["date"]) |
        (data["away_elo_date_used"] > data["date"]) |
        (data["home_elo"] <= 0) |
        (data["away_elo"] <= 0)
    )
].copy()

data_clean["elo_advantage"] = data_clean["home_elo"] - data_clean["away_elo"]

print("Rows before:", len(data))
print("Rows after:", len(data_clean))

data_clean.to_csv("../data/matches_with_fifa_rankings_and_elo_filled_clean.csv", index=False)

Rows before: 1065
Rows after: 1054


In [229]:
data_clean.isnull().sum()

date                          0
home_team                     0
away_team                     0
home_score                    0
away_score                    0
tournament                    0
city                          0
country                       0
neutral                       0
year                          0
is_world_cup                  0
is_qualifier                  0
match_id                      0
home_rank_date_used           0
home_rank                     0
home_previous_rank            0
home_rank_change              0
home_total_points             0
home_previous_points          0
away_rank_date_used           0
away_rank                     0
away_previous_rank            0
away_rank_change              0
away_total_points             0
away_previous_points          0
rank_advantage                0
points_diff                   0
result                        0
home_elo_date_used            0
home_elo_rank               684
home_elo                      0
home_elo

In [230]:
elo_columns_to_drop = [
    "home_elo_rank",
    "home_elo_avg_rank",
    "home_elo_avg_rating",
    "home_elo_matches_total",
    "home_elo_wins",
    "home_elo_losses",
    "home_elo_draws",
    "home_elo_goals_for",
    "home_elo_goals_against",
    "home_is_host",

    "away_elo_rank",
    "away_elo_avg_rank",
    "away_elo_avg_rating",
    "away_elo_matches_total",
    "away_elo_wins",
    "away_elo_losses",
    "away_elo_draws",
    "away_elo_goals_for",
    "away_elo_goals_against",
    "away_is_host",

    "elo_rank_advantage",
    "elo_avg_rating_advantage",
    "elo_win_rate_home",
    "elo_win_rate_away",
    "elo_win_rate_advantage",

    "home_elo_was_filled",
    "away_elo_was_filled",
    "home_elo_date_used",
    "away_elo_date_used"  
]

data_clean = data_clean.drop(columns=elo_columns_to_drop, errors='ignore')
data_clean.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_previous_rank,away_rank_change,away_total_points,away_previous_points,rank_advantage,points_diff,result,home_elo,away_elo,elo_advantage
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,14.0,0.0,1638.30,1638.76,-43.0,-226.17,-1,1578.0,1839.0,-261.0
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,5.0,0.0,1750.51,1750.51,-19.0,-207.09,-1,1813.0,2101.0,-288.0
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,17.0,0.0,1596.66,1596.66,-26.0,-142.14,-1,1729.0,1847.0,-118.0
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,2.0,0.0,1826.35,1826.35,-44.0,-378.08,0,1849.0,2149.0,-300.0
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,78.0,-1.0,1306.01,1307.09,28.0,128.70,1,1629.0,1496.0,133.0


In [231]:
data_clean = data_clean[['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'year', 'is_world_cup',
       'is_qualifier', 'match_id', 'home_rank_date_used', 'home_rank',
       'home_previous_rank', 'home_rank_change', 'home_total_points',
       'home_previous_points', 'away_rank_date_used', 'away_rank',
       'away_previous_rank', 'away_rank_change', 'away_total_points',
       'away_previous_points', 'rank_advantage', 'points_diff',
       'home_elo', 'away_elo', 'elo_advantage', 'result']]
data_clean.head()

data_clean.to_csv('../data/fifa_and_elo_rankings_clean_filled.csv')

In [232]:
prediction_data = pd.read_csv('../data/Processed_Project_Data/prediction_data_2026.csv')
prediction_data.head()

,match_id,date,home_team,away_team,tournament,city,country,neutral,year,is_world_cup,...,home_elo,away_elo,elo_advantage,home_elo_rank,away_elo_rank,elo_rank_advantage,home_team_fifa_name,away_team_fifa_name,home_team_elo_name,away_team_elo_name
0,0,2026-06-11,Mexico,South Africa,FIFA World Cup,Mexico City,Mexico,0,2026,1,...,1875,1517,358,18,80,62,Mexico,South Africa,Mexico,South Africa
1,1,2026-06-11,South Korea,Czech Republic,FIFA World Cup,Zapopan,Mexico,1,2026,1,...,1758,1740,18,33,35,2,Korea Republic,Czechia,South Korea,Czechia
2,2,2026-06-12,Canada,Bosnia and Herzegovina,FIFA World Cup,Toronto,Canada,0,2026,1,...,1788,1595,193,25,65,40,Canada,Bosnia and Herzegovina,Canada,Bosnia and Herzegovina
3,3,2026-06-12,United States,Paraguay,FIFA World Cup,Inglewood,United States,0,2026,1,...,1726,1834,-108,39,22,-17,USA,Paraguay,United States,Paraguay
4,4,2026-06-13,Qatar,Switzerland,FIFA World Cup,Santa Clara,United States,1,2026,1,...,1421,1891,-470,96,17,-79,Qatar,Switzerland,Qatar,Switzerland


In [238]:
prediction_data.columns.unique()

prediction_data.drop(columns=[
     "home_team_fifa_name",
     "away_team_fifa_name",
     "home_team_elo_name",
     "away_team_elo_name"
    ], inplace=True
)
prediction_data.to_csv('../data/Processed_Project_Data/prediction_data_2026.csv')

In [245]:
## Final check of the data
train_ds = pd.read_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')
train_ds.columns.unique()
train_ds.drop(columns=['Unnamed: 0'], inplace=True)
train_ds.to_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')

In [248]:
test_ds = pd.read_csv("../data/Processed_Project_Data/prediction_data_2026.csv")
test_ds.drop(columns=['Unnamed: 0'], inplace=True)
test_ds.to_csv('../data/Processed_Project_Data/prediction_data_2026.csv')

In [ ]:
test_ds

,match_id,date,home_team,away_team,tournament,city,country,neutral,year,is_world_cup,...,away_total_points,points_diff,home_elo_date_used,away_elo_date_used,home_elo,away_elo,elo_advantage,home_elo_rank,away_elo_rank,elo_rank_advantage
0,0,2026-06-11,Mexico,South Africa,FIFA World Cup,Mexico City,Mexico,0,2026,1,...,1429.73,251.30,2026-06-11,2026-06-11,1875,1517,358,18,80,62
1,1,2026-06-11,South Korea,Czech Republic,FIFA World Cup,Zapopan,Mexico,1,2026,1,...,1513.74,74.93,2026-06-11,2026-06-11,1758,1740,18,33,35,2
2,2,2026-06-12,Canada,Bosnia and Herzegovina,FIFA World Cup,Toronto,Canada,0,2026,1,...,1398.23,158.25,2026-06-11,2026-06-11,1788,1595,193,25,65,40
3,3,2026-06-12,United States,Paraguay,FIFA World Cup,Inglewood,United States,0,2026,1,...,1503.51,169.62,2026-06-11,2026-06-11,1726,1834,-108,39,22,-17
4,4,2026-06-13,Qatar,Switzerland,FIFA World Cup,Santa Clara,United States,1,2026,1,...,1649.40,-194.44,2026-06-11,2026-06-11,1421,1891,-470,96,17,-79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,67,2026-06-27,Jordan,Argentina,FIFA World Cup,Arlington,United States,1,2026,1,...,1874.82,-483.37,2026-06-11,2026-06-11,1680,2115,-435,52,2,-50
68,68,2026-06-27,Colombia,Portugal,FIFA World Cup,Miami Gardens,United States,1,2026,1,...,1763.83,-70.74,2026-06-11,2026-06-11,1982,1989,-7,7,6,-1
69,69,2026-06-27,DR Congo,Uzbekistan,FIFA World Cup,Atlanta,United States,1,2026,1,...,1469.40,10.24,2026-06-11,2026-06-11,1652,1714,-62,55,42,-13
70,70,2026-06-27,Panama,England,FIFA World Cup,East Rutherford,United States,1,2026,1,...,1825.97,-285.33,2026-06-11,2026-06-11,1730,2024,-294,38,4,-34


In [250]:
train_ds

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_previous_rank,away_rank_change,away_total_points,away_previous_points,rank_advantage,points_diff,home_elo,away_elo,elo_advantage,result
0,2022-01-27,Jamaica,Mexico,1.0,2.0,FIFA World Cup qualification,Kingston,Jamaica,0,2022,...,14.0,0.0,1638.30,1638.76,-43.0,-226.17,1578.0,1839.0,-261.0,-1
1,2022-01-27,Chile,Argentina,1.0,2.0,FIFA World Cup qualification,Calama,Chile,0,2022,...,5.0,0.0,1750.51,1750.51,-19.0,-207.09,1813.0,2101.0,-288.0,-1
2,2022-01-27,Paraguay,Uruguay,0.0,1.0,FIFA World Cup qualification,Asunción,Paraguay,0,2022,...,17.0,0.0,1596.66,1596.66,-26.0,-142.14,1729.0,1847.0,-118.0,-1
3,2022-01-27,Ecuador,Brazil,1.0,1.0,FIFA World Cup qualification,Quito,Ecuador,0,2022,...,2.0,0.0,1826.35,1826.35,-44.0,-378.08,1849.0,2149.0,-300.0,0
4,2022-01-27,Saudi Arabia,Oman,1.0,0.0,FIFA World Cup qualification,Jeddah,Saudi Arabia,0,2022,...,78.0,-1.0,1306.01,1307.09,28.0,128.70,1629.0,1496.0,133.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1049,2026-03-31,DR Congo,Jamaica,1.0,0.0,FIFA World Cup qualification,Zapopan,Mexico,1,2026,...,70.0,1.0,1377.21,1363.00,21.0,91.01,1657.0,1527.0,130.0,1
1050,2026-03-31,Iraq,Bolivia,2.0,1.0,FIFA World Cup qualification,Guadalupe,Mexico,1,2026,...,76.0,0.0,1330.52,1330.00,18.0,106.42,1582.0,1675.0,-93.0,1
1051,2026-03-31,Bosnia and Herzegovina,Italy,1.0,1.0,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,0,2026,...,12.0,-1.0,1702.06,1702.00,-58.0,-339.69,1572.0,1859.0,-287.0,0
1052,2026-03-31,Sweden,Poland,3.0,2.0,FIFA World Cup qualification,Solna,Sweden,0,2026,...,31.0,-3.0,1532.04,1532.00,-8.0,-44.91,1660.0,1735.0,-75.0,1


# Adding post-2022 continental tournament matches

Adds matches from **UEFA Euro**, **African Cup of Nations**, **AFC Asian Cup** and **Copa America** played after 2022, for teams that qualified for the 2026 World Cup.

Follows the exact same steps as above: FIFA ranking merge (time-aware `merge_asof`), ELO merge (current snapshot + historical fallback fill), feature computation (`rank_advantage`, `points_diff`, `elo_advantage`, `result`), and the same cleaning rules before appending to `fifa_and_elo_rankings_clean_filled.csv`.

In [ ]:
import pandas as pd

## 0. Config

In [ ]:
WC2026_TEAMS = [
    'Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina',
    'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic',
    'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti',
    'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands',
    'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia',
    'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland',
    'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan'
]

TOURNAMENTS = ['UEFA Euro', 'African Cup of Nations', 'AFC Asian Cup', 'Copa América']

## 1. Load existing final dataset

In [ ]:
final_existing = pd.read_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')
final_existing = final_existing.drop(columns=["Unnamed: 0"], errors="ignore")
max_match_id = final_existing["match_id"].max()

## 2. Select new matches from results.csv

In [ ]:
results = pd.read_csv('../data/raw_data/results.csv')
results["date"] = pd.to_datetime(results["date"])

new_matches = results[
    (results["date"] > "2022-12-31") &
    (results["tournament"].isin(TOURNAMENTS)) &
    (results["home_team"].isin(WC2026_TEAMS) | results["away_team"].isin(WC2026_TEAMS))
].copy()

new_matches["neutral"] = new_matches["neutral"].map({True: 1, False: 0, "TRUE": 1, "FALSE": 0})
new_matches["neutral"] = new_matches["neutral"].fillna(new_matches["neutral"]).astype(int)
new_matches["year"] = new_matches["date"].dt.year
new_matches["is_world_cup"] = 0
new_matches["is_qualifier"] = 0

new_matches = new_matches.sort_values(["date", "home_team"]).reset_index(drop=True)
new_matches["match_id"] = range(max_match_id + 1, max_match_id + 1 + len(new_matches))

print(f"New matches selected: {len(new_matches)}")
print(new_matches["tournament"].value_counts())

## 3. FIFA ranking features

In [ ]:
rank_data = pd.read_csv('../data/ProcessedDataForTestModel/FIFA_rankings_2021_2026.csv')
rank_data = rank_data.drop(columns=["Unnamed: 0"], errors="ignore")
rank_data["rank_date"] = pd.to_datetime(rank_data["rank_date"])

TEAM_NAME_FIXES = {
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "Kyrgyzstan": "Kyrgyz Republic",
    "North Korea": "Korea DPR",
    "South Korea": "Korea Republic",
    "Taiwan": "Chinese Taipei",
    "United States": "USA",
    "United States Virgin Islands": "US Virgin Islands",
    "Saint Kitts and Nevis": "St Kitts and Nevis",
    "Saint Lucia": "St Lucia",
    "Saint Vincent and the Grenadines": "St Vincent and the Grenadines",
}

new_matches["home_team_rank_name"] = new_matches["home_team"].replace(TEAM_NAME_FIXES)
new_matches["away_team_rank_name"] = new_matches["away_team"].replace(TEAM_NAME_FIXES)

ranking_data = rank_data[
    [
        "rank_date",
        "country",
        "country_code",
        "rank",
        "previous_rank",
        "rank_change",
        "total_points",
        "previous_points"
    ]
].copy()

ranking_data = ranking_data.rename(columns={"country": "team_rank_name"})


def add_ranking_to_matches(matches_df, team_col, prefix):
    left = matches_df[["match_id", "date", team_col]].copy()
    left = left.rename(columns={team_col: "team_rank_name"})

    right = ranking_data.copy()

    left = left.sort_values(["date", "team_rank_name"])
    right = right.sort_values(["rank_date", "team_rank_name"])

    merged = pd.merge_asof(
        left, right,
        left_on="date", right_on="rank_date",
        by="team_rank_name", direction="backward"
    )

    merged = merged.rename(columns={
        "rank_date": f"{prefix}_rank_date_used",
        "country_code": f"{prefix}_country_code",
        "rank": f"{prefix}_rank",
        "previous_rank": f"{prefix}_previous_rank",
        "rank_change": f"{prefix}_rank_change",
        "total_points": f"{prefix}_total_points",
        "previous_points": f"{prefix}_previous_points"
    })

    return merged[[
        "match_id",
        f"{prefix}_rank_date_used", f"{prefix}_country_code", f"{prefix}_rank",
        f"{prefix}_previous_rank", f"{prefix}_rank_change",
        f"{prefix}_total_points", f"{prefix}_previous_points"
    ]]


home_ranking = add_ranking_to_matches(new_matches, "home_team_rank_name", "home")
away_ranking = add_ranking_to_matches(new_matches, "away_team_rank_name", "away")

new_matches = new_matches.merge(home_ranking, on="match_id", how="left")
new_matches = new_matches.merge(away_ranking, on="match_id", how="left")

new_matches["rank_advantage"] = new_matches["away_rank"] - new_matches["home_rank"]
new_matches["points_diff"] = new_matches["home_total_points"] - new_matches["away_total_points"]


def get_match_result(row):
    if row["home_score"] > row["away_score"]:
        return 1
    elif row["home_score"] < row["away_score"]:
        return -1
    else:
        return 0


new_matches["result"] = new_matches.apply(get_match_result, axis=1)

missing_rankings = new_matches[
    new_matches["home_rank"].isna() | new_matches["away_rank"].isna()
][["date", "home_team", "away_team", "home_rank", "away_rank"]]
print("\nMatches with missing FIFA ranking:", len(missing_rankings))
print(missing_rankings)

new_matches = new_matches.drop(
    columns=["home_team_rank_name", "away_team_rank_name", "home_country_code", "away_country_code"],
    errors="ignore"
)

## 4. ELO features (current snapshot)

In [ ]:
elo_data = pd.read_csv('../data/raw_data/elo_ratings_wc2026.csv')
elo_data["snapshot_date"] = pd.to_datetime(elo_data["snapshot_date"])

elo_data = elo_data[[
    "snapshot_date", "country", "rank", "rating", "rank_avg", "rating_avg",
    "matches_total", "wins", "losses", "draws", "goals_for", "goals_against",
    "confederation", "is_host"
]].copy()

elo_data = elo_data.rename(columns={
    "snapshot_date": "elo_date",
    "country": "team_elo_name",
    "rank": "elo_rank",
    "rating": "elo_rating",
    "rank_avg": "elo_avg_rank",
    "rating_avg": "elo_avg_rating"
})

elo_data = elo_data[elo_data["elo_date"] >= "2021-01-01"].copy()

TEAM_NAME_FIXES_ELO = {
    "USA": "United States",
    "United States": "United States",

    "IR Iran": "Iran",
    "Iran": "Iran",

    "Korea Republic": "South Korea",
    "South Korea": "South Korea",

    "Korea DPR": "North Korea",
    "North Korea": "North Korea",

    "Congo DR": "DR Congo",
    "DR Congo": "DR Congo",

    "Côte d'Ivoire": "Ivory Coast",
    "Ivory Coast": "Ivory Coast",

    "Cabo Verde": "Cape Verde",
    "Cape Verde": "Cape Verde",

    "Kyrgyz Republic": "Kyrgyzstan",
    "Kyrgyzstan": "Kyrgyzstan",
}

new_matches["home_team_elo_name"] = new_matches["home_team"].replace(TEAM_NAME_FIXES_ELO)
new_matches["away_team_elo_name"] = new_matches["away_team"].replace(TEAM_NAME_FIXES_ELO)


def add_elo_to_matches(matches_df, team_col, prefix):
    left = matches_df[["match_id", "date", team_col]].copy()
    left = left.rename(columns={team_col: "team_elo_name"})

    right = elo_data.copy()

    left = left.sort_values(["date", "team_elo_name"])
    right = right.sort_values(["elo_date", "team_elo_name"])

    merged = pd.merge_asof(
        left, right,
        left_on="date", right_on="elo_date",
        by="team_elo_name", direction="backward"
    )

    merged = merged.rename(columns={
        "elo_date": f"{prefix}_elo_date_used",
        "elo_rank": f"{prefix}_elo_rank",
        "elo_rating": f"{prefix}_elo",
        "elo_avg_rank": f"{prefix}_elo_avg_rank",
        "elo_avg_rating": f"{prefix}_elo_avg_rating",
        "matches_total": f"{prefix}_elo_matches_total",
        "wins": f"{prefix}_elo_wins",
        "losses": f"{prefix}_elo_losses",
        "draws": f"{prefix}_elo_draws",
        "goals_for": f"{prefix}_elo_goals_for",
        "goals_against": f"{prefix}_elo_goals_against",
        "confederation": f"{prefix}_confederation",
        "is_host": f"{prefix}_is_host"
    })

    return merged[[
        "match_id",
        f"{prefix}_elo_date_used", f"{prefix}_elo_rank", f"{prefix}_elo",
        f"{prefix}_elo_avg_rank", f"{prefix}_elo_avg_rating", f"{prefix}_elo_matches_total",
        f"{prefix}_elo_wins", f"{prefix}_elo_losses", f"{prefix}_elo_draws",
        f"{prefix}_elo_goals_for", f"{prefix}_elo_goals_against",
        f"{prefix}_confederation", f"{prefix}_is_host"
    ]]


home_elo = add_elo_to_matches(new_matches, "home_team_elo_name", "home")
away_elo = add_elo_to_matches(new_matches, "away_team_elo_name", "away")

new_matches = new_matches.merge(home_elo, on="match_id", how="left")
new_matches = new_matches.merge(away_elo, on="match_id", how="left")

new_matches["elo_advantage"] = new_matches["home_elo"] - new_matches["away_elo"]
new_matches["elo_rank_advantage"] = new_matches["away_elo_rank"] - new_matches["home_elo_rank"]
new_matches["elo_avg_rating_advantage"] = new_matches["home_elo_avg_rating"] - new_matches["away_elo_avg_rating"]
new_matches["elo_win_rate_home"] = new_matches["home_elo_wins"] / new_matches["home_elo_matches_total"]
new_matches["elo_win_rate_away"] = new_matches["away_elo_wins"] / new_matches["away_elo_matches_total"]
new_matches["elo_win_rate_advantage"] = new_matches["elo_win_rate_home"] - new_matches["elo_win_rate_away"]

missing_elo = new_matches[
    new_matches["home_elo"].isna() | new_matches["away_elo"].isna()
][["date", "home_team", "away_team", "home_elo", "away_elo"]]
print("\nMatches with missing ELO (before historical fallback):", len(missing_elo))

new_matches = new_matches.drop(
    columns=["home_team_elo_name", "away_team_elo_name", "home_confederation", "away_confederation"],
    errors="ignore"
)

## 5. ELO historical fallback fill

In [ ]:
import re

elo_hist = pd.read_csv('../data/raw_data/eloratings.csv')
elo_hist["date"] = pd.to_datetime(elo_hist["date"], format="mixed", errors="coerce")


def clean_team_name(name):
    if pd.isna(name):
        return name
    name = str(name)
    name = name.replace("\xa0", " ")
    name = re.sub(r"\s+", " ", name)
    return name.strip()


new_matches["home_team_elo_name_fill"] = new_matches["home_team"].apply(clean_team_name)
new_matches["away_team_elo_name_fill"] = new_matches["away_team"].apply(clean_team_name)
elo_hist["team_elo_name"] = elo_hist["team"].apply(clean_team_name)

TEAM_NAME_FIXES_ELO_HIST = {
    "American Samoa": "Eastern Samoa",
    "China PR": "China",
    "Czech Republic": "Czechia",
    "DR Congo": "Democratic Republic of Congo",
    "Macau": "Macao",
    "Republic of Ireland": "Ireland",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Timor-Leste": "East Timor",
    "United States Virgin Islands": "US Virgin Islands",

    "USA": "United States",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "IR Iran": "Iran",
    "Congo DR": "Democratic Republic of Congo",
    "Côte d'Ivoire": "Ivory Coast",
    "Cabo Verde": "Cape Verde",
    "Kyrgyz Republic": "Kyrgyzstan",
    "UAE": "United Arab Emirates",
}

new_matches["home_team_elo_name_fill"] = new_matches["home_team_elo_name_fill"].replace(TEAM_NAME_FIXES_ELO_HIST)
new_matches["away_team_elo_name_fill"] = new_matches["away_team_elo_name_fill"].replace(TEAM_NAME_FIXES_ELO_HIST)

elo_hist = elo_hist[["date", "team_elo_name", "rating", "change"]].copy()
elo_hist = elo_hist.rename(columns={"date": "elo_date", "rating": "elo_rating", "change": "elo_change"})
elo_hist = elo_hist.dropna(subset=["elo_date", "elo_rating"])
elo_hist = elo_hist.sort_values(["elo_date", "team_elo_name"]).drop_duplicates(
    ["team_elo_name", "elo_date"], keep="last"
)


def get_historical_elo(matches_df, team_col, prefix):
    left = matches_df[["match_id", "date", team_col]].copy()
    left = left.rename(columns={team_col: "team_elo_name"})

    left = left.sort_values(["date", "team_elo_name"]).reset_index(drop=True)
    right = elo_hist.sort_values(["elo_date", "team_elo_name"]).reset_index(drop=True)

    previous_elo = pd.merge_asof(
        left, right,
        left_on="date", right_on="elo_date",
        by="team_elo_name", direction="backward",
        allow_exact_matches=False
    )
    previous_elo["elo_source_fill"] = "historical_previous"

    future_elo = pd.merge_asof(
        left, right,
        left_on="date", right_on="elo_date",
        by="team_elo_name", direction="forward",
        allow_exact_matches=True
    )

    missing_mask = previous_elo["elo_rating"].isna()

    previous_elo.loc[missing_mask, "elo_date"] = future_elo.loc[missing_mask, "elo_date"]
    previous_elo.loc[missing_mask, "elo_rating"] = future_elo.loc[missing_mask, "elo_rating"]
    previous_elo.loc[missing_mask, "elo_change"] = future_elo.loc[missing_mask, "elo_change"]

    previous_elo.loc[
        missing_mask & future_elo["elo_rating"].notna(), "elo_source_fill"
    ] = "historical_future_fallback"

    previous_elo = previous_elo.rename(columns={
        "elo_date": f"{prefix}_elo_date_fill",
        "elo_rating": f"{prefix}_elo_fill",
        "elo_change": f"{prefix}_elo_change_fill",
        "elo_source_fill": f"{prefix}_elo_source_fill"
    })

    return previous_elo[[
        "match_id", f"{prefix}_elo_date_fill", f"{prefix}_elo_fill",
        f"{prefix}_elo_change_fill", f"{prefix}_elo_source_fill"
    ]]


home_fill = get_historical_elo(new_matches, "home_team_elo_name_fill", "home")
away_fill = get_historical_elo(new_matches, "away_team_elo_name_fill", "away")

new_matches = new_matches.merge(home_fill, on="match_id", how="left")
new_matches = new_matches.merge(away_fill, on="match_id", how="left")

new_matches["home_elo"] = new_matches["home_elo"].fillna(new_matches["home_elo_fill"])
new_matches["away_elo"] = new_matches["away_elo"].fillna(new_matches["away_elo_fill"])
new_matches["home_elo_date_used"] = new_matches["home_elo_date_used"].fillna(new_matches["home_elo_date_fill"])
new_matches["away_elo_date_used"] = new_matches["away_elo_date_used"].fillna(new_matches["away_elo_date_fill"])

new_matches["elo_advantage"] = new_matches["home_elo"] - new_matches["away_elo"]

print("\nHome ELO still missing:", new_matches["home_elo"].isna().sum())
print("Away ELO still missing:", new_matches["away_elo"].isna().sum())

temp_cols = [
    "home_team_elo_name_fill", "away_team_elo_name_fill",
    "home_elo_date_fill", "home_elo_fill", "home_elo_change_fill", "home_elo_source_fill",
    "away_elo_date_fill", "away_elo_fill", "away_elo_change_fill", "away_elo_source_fill",
]
new_matches = new_matches.drop(columns=temp_cols, errors="ignore")

## 6. Drop bad rows (same cleaning as original pipeline)

In [ ]:
new_matches["date"] = pd.to_datetime(new_matches["date"])
new_matches["home_elo_date_used"] = pd.to_datetime(new_matches["home_elo_date_used"])
new_matches["away_elo_date_used"] = pd.to_datetime(new_matches["away_elo_date_used"])

before = len(new_matches)

new_matches_clean = new_matches[
    ~(
        (new_matches["home_elo_date_used"] > new_matches["date"]) |
        (new_matches["away_elo_date_used"] > new_matches["date"]) |
        new_matches["home_rank"].isna() |
        new_matches["away_rank"].isna() |
        (new_matches["home_elo"] <= 0) |
        (new_matches["away_elo"] <= 0) |
        new_matches["home_elo"].isna() |
        new_matches["away_elo"].isna()
    )
].copy()

new_matches_clean["elo_advantage"] = new_matches_clean["home_elo"] - new_matches_clean["away_elo"]

print(f"\nRows before cleaning: {before}")
print(f"Rows after cleaning: {len(new_matches_clean)}")

## 7. Select final columns and append

In [ ]:
FINAL_COLUMNS = [
    'date', 'home_team', 'away_team', 'home_score', 'away_score',
    'tournament', 'city', 'country', 'neutral', 'year', 'is_world_cup',
    'is_qualifier', 'match_id', 'home_rank_date_used', 'home_rank',
    'home_previous_rank', 'home_rank_change', 'home_total_points',
    'home_previous_points', 'away_rank_date_used', 'away_rank',
    'away_previous_rank', 'away_rank_change', 'away_total_points',
    'away_previous_points', 'rank_advantage', 'points_diff',
    'home_elo', 'away_elo', 'elo_advantage', 'result'
]

new_matches_final = new_matches_clean[FINAL_COLUMNS].copy()
new_matches_final["match_id"] = range(max_match_id + 1, max_match_id + 1 + len(new_matches_final))

combined = pd.concat([final_existing[FINAL_COLUMNS], new_matches_final], ignore_index=True)

combined.to_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')

print(f"\nFinal dataset rows: {len(combined)} (was {len(final_existing)}, added {len(new_matches_final)})")
print("Saved: ../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv")

In [1]:
import pandas as pd
df = pd.read_csv('../data/Processed_Project_Data/fifa_and_elo_rankings_clean_filled.csv')
print('Ukupno redova:', len(df))
print(df['tournament'].value_counts())
df.tail(10)[['date','home_team','away_team','tournament','result']]


Ukupno redova: 1239
tournament
FIFA World Cup qualification    990
African Cup of Nations           73
FIFA World Cup                   64
UEFA Euro                        44
AFC Asian Cup                    38
Copa América                     30
Name: count, dtype: int64


,date,home_team,away_team,tournament,result
1229,2026-01-06 00:00:00,Algeria,DR Congo,African Cup of Nations,1
1230,2026-01-06 00:00:00,Ivory Coast,Burkina Faso,African Cup of Nations,1
1231,2026-01-09 00:00:00,Mali,Senegal,African Cup of Nations,-1
1232,2026-01-09 00:00:00,Morocco,Cameroon,African Cup of Nations,1
1233,2026-01-10 00:00:00,Algeria,Nigeria,African Cup of Nations,-1
1234,2026-01-10 00:00:00,Egypt,Ivory Coast,African Cup of Nations,1
1235,2026-01-14 00:00:00,Morocco,Nigeria,African Cup of Nations,0
1236,2026-01-14 00:00:00,Senegal,Egypt,African Cup of Nations,1
1237,2026-01-17 00:00:00,Egypt,Nigeria,African Cup of Nations,0
1238,2026-01-18 00:00:00,Morocco,Senegal,African Cup of Nations,1
